In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [2]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
len(words)

32033

In [4]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [17]:
# build the dataset

block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words[:5]:
    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix= stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [18]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [20]:
X, Y

(tensor([[ 0,  0,  0],
         [ 0,  0,  5],
         [ 0,  5, 13],
         [ 5, 13, 13],
         [13, 13,  1],
         [ 0,  0,  0],
         [ 0,  0, 15],
         [ 0, 15, 12],
         [15, 12,  9],
         [12,  9, 22],
         [ 9, 22,  9],
         [22,  9,  1],
         [ 0,  0,  0],
         [ 0,  0,  1],
         [ 0,  1, 22],
         [ 1, 22,  1],
         [ 0,  0,  0],
         [ 0,  0,  9],
         [ 0,  9, 19],
         [ 9, 19,  1],
         [19,  1,  2],
         [ 1,  2,  5],
         [ 2,  5, 12],
         [ 5, 12, 12],
         [12, 12,  1],
         [ 0,  0,  0],
         [ 0,  0, 19],
         [ 0, 19, 15],
         [19, 15, 16],
         [15, 16,  8],
         [16,  8,  9],
         [ 8,  9,  1]]),
 tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
          1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0]))

In [ ]:
# embedding lookup table C:
# in the paper: 17K words in 30 dim spqce, Here as we have only 27, ok to start with a 2 dim embedding


In [21]:
C = torch.randn((27, 2))

In [23]:
C

tensor([[-0.0261,  0.8148],
        [ 0.9249,  0.5281],
        [-0.2078,  0.5555],
        [ 1.7961,  1.8419],
        [-0.1946,  0.0130],
        [ 0.0865,  1.6616],
        [-0.4026,  1.4530],
        [-0.0779, -0.4773],
        [ 0.2206, -0.3968],
        [ 1.2898,  0.4652],
        [-0.0048, -0.4839],
        [-0.6401,  1.0450],
        [ 0.4949,  0.9084],
        [-0.6014,  1.4995],
        [-0.3658,  0.2426],
        [ 0.6220, -0.2713],
        [-0.0629, -1.6374],
        [-0.1054, -0.1888],
        [-1.4285,  0.0168],
        [-1.1166, -1.0940],
        [ 0.1538, -1.5926],
        [ 0.9553, -0.3261],
        [-0.3351,  0.1436],
        [-0.6428, -1.1453],
        [ 0.8212,  1.8749],
        [ 0.6095,  0.8392],
        [-0.1775, -1.1901]])

In [35]:
C.shape

torch.Size([27, 2])

In [22]:
# before embedding all integers inside the input X, we make an example with 5
# one way is to index 5 in the lookup table C, getting the 5th row
C[5]

tensor([0.0865, 1.6616])

In [29]:
# the other way is to use one-hot encoding. Output is identical. All zero masking out all the rows but the 5th
F.one_hot(torch.tensor(5), num_classes=27).float() @ C

tensor([0.0865, 1.6616])

In [ ]:
 # using indexing to do embedding is equal to use one-hot encoding. Here we use index as it is much faster.
# this can be seen as first layer of NN.

In [31]:
emb = C[X]

In [34]:
emb.shape

torch.Size([32, 3, 2])

In [33]:
X, C, emb

(tensor([[ 0,  0,  0],
         [ 0,  0,  5],
         [ 0,  5, 13],
         [ 5, 13, 13],
         [13, 13,  1],
         [ 0,  0,  0],
         [ 0,  0, 15],
         [ 0, 15, 12],
         [15, 12,  9],
         [12,  9, 22],
         [ 9, 22,  9],
         [22,  9,  1],
         [ 0,  0,  0],
         [ 0,  0,  1],
         [ 0,  1, 22],
         [ 1, 22,  1],
         [ 0,  0,  0],
         [ 0,  0,  9],
         [ 0,  9, 19],
         [ 9, 19,  1],
         [19,  1,  2],
         [ 1,  2,  5],
         [ 2,  5, 12],
         [ 5, 12, 12],
         [12, 12,  1],
         [ 0,  0,  0],
         [ 0,  0, 19],
         [ 0, 19, 15],
         [19, 15, 16],
         [15, 16,  8],
         [16,  8,  9],
         [ 8,  9,  1]]),
 tensor([[-0.0261,  0.8148],
         [ 0.9249,  0.5281],
         [-0.2078,  0.5555],
         [ 1.7961,  1.8419],
         [-0.1946,  0.0130],
         [ 0.0865,  1.6616],
         [-0.4026,  1.4530],
         [-0.0779, -0.4773],
         [ 0.2206, -0.3968],
 

In [36]:
emb

tensor([[[-0.0261,  0.8148],
         [-0.0261,  0.8148],
         [-0.0261,  0.8148]],

        [[-0.0261,  0.8148],
         [-0.0261,  0.8148],
         [ 0.0865,  1.6616]],

        [[-0.0261,  0.8148],
         [ 0.0865,  1.6616],
         [-0.6014,  1.4995]],

        [[ 0.0865,  1.6616],
         [-0.6014,  1.4995],
         [-0.6014,  1.4995]],

        [[-0.6014,  1.4995],
         [-0.6014,  1.4995],
         [ 0.9249,  0.5281]],

        [[-0.0261,  0.8148],
         [-0.0261,  0.8148],
         [-0.0261,  0.8148]],

        [[-0.0261,  0.8148],
         [-0.0261,  0.8148],
         [ 0.6220, -0.2713]],

        [[-0.0261,  0.8148],
         [ 0.6220, -0.2713],
         [ 0.4949,  0.9084]],

        [[ 0.6220, -0.2713],
         [ 0.4949,  0.9084],
         [ 1.2898,  0.4652]],

        [[ 0.4949,  0.9084],
         [ 1.2898,  0.4652],
         [-0.3351,  0.1436]],

        [[ 1.2898,  0.4652],
         [-0.3351,  0.1436],
         [ 1.2898,  0.4652]],

        [[-0.3351,  0

In [40]:
torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1)

tensor([[-0.0261,  0.8148, -0.0261,  0.8148, -0.0261,  0.8148],
        [-0.0261,  0.8148, -0.0261,  0.8148,  0.0865,  1.6616],
        [-0.0261,  0.8148,  0.0865,  1.6616, -0.6014,  1.4995],
        [ 0.0865,  1.6616, -0.6014,  1.4995, -0.6014,  1.4995],
        [-0.6014,  1.4995, -0.6014,  1.4995,  0.9249,  0.5281],
        [-0.0261,  0.8148, -0.0261,  0.8148, -0.0261,  0.8148],
        [-0.0261,  0.8148, -0.0261,  0.8148,  0.6220, -0.2713],
        [-0.0261,  0.8148,  0.6220, -0.2713,  0.4949,  0.9084],
        [ 0.6220, -0.2713,  0.4949,  0.9084,  1.2898,  0.4652],
        [ 0.4949,  0.9084,  1.2898,  0.4652, -0.3351,  0.1436],
        [ 1.2898,  0.4652, -0.3351,  0.1436,  1.2898,  0.4652],
        [-0.3351,  0.1436,  1.2898,  0.4652,  0.9249,  0.5281],
        [-0.0261,  0.8148, -0.0261,  0.8148, -0.0261,  0.8148],
        [-0.0261,  0.8148, -0.0261,  0.8148,  0.9249,  0.5281],
        [-0.0261,  0.8148,  0.9249,  0.5281, -0.3351,  0.1436],
        [ 0.9249,  0.5281, -0.3351,  0.1

In [58]:
W1 = torch.randn((6, 100)) 
b1 = torch.randn(100)
# hidden layer construction - initialized with random parameters
# 6 is the number of inputs given from 3 vectors projected on 2D spaced (2-dim embedding)
# 100 is the number of neurons and we can chose by us 

In [64]:
# problem: emb @ W1 + b1 cannot be done as emb is torch.Size([32, 3, 2]) and W1 is (6, 100)
# in emb, 3 vector concatenated (torch.cat)so that emb becomes (32, 6)
# the definition above is not generic and need to be rewritten if block size changes
# a generic espression is the following:
# len(torch.unbind(emb,1)) is 3 -> returning 3 tensors: one for index 0, one for index 1 and one for index 2
#torch.cat(torch.unbind(emb,1), 1).shape
torch.unbind(emb,1)



(tensor([[-0.0261,  0.8148],
         [-0.0261,  0.8148],
         [-0.0261,  0.8148],
         [ 0.0865,  1.6616],
         [-0.6014,  1.4995],
         [-0.0261,  0.8148],
         [-0.0261,  0.8148],
         [-0.0261,  0.8148],
         [ 0.6220, -0.2713],
         [ 0.4949,  0.9084],
         [ 1.2898,  0.4652],
         [-0.3351,  0.1436],
         [-0.0261,  0.8148],
         [-0.0261,  0.8148],
         [-0.0261,  0.8148],
         [ 0.9249,  0.5281],
         [-0.0261,  0.8148],
         [-0.0261,  0.8148],
         [-0.0261,  0.8148],
         [ 1.2898,  0.4652],
         [-1.1166, -1.0940],
         [ 0.9249,  0.5281],
         [-0.2078,  0.5555],
         [ 0.0865,  1.6616],
         [ 0.4949,  0.9084],
         [-0.0261,  0.8148],
         [-0.0261,  0.8148],
         [-0.0261,  0.8148],
         [-1.1166, -1.0940],
         [ 0.6220, -0.2713],
         [-0.0629, -1.6374],
         [ 0.2206, -0.3968]]),
 tensor([[-0.0261,  0.8148],
         [-0.0261,  0.8148],
         [ 0

In [65]:
torch.cat(torch.unbind(emb,1), 1)

tensor([[-0.0261,  0.8148, -0.0261,  0.8148, -0.0261,  0.8148],
        [-0.0261,  0.8148, -0.0261,  0.8148,  0.0865,  1.6616],
        [-0.0261,  0.8148,  0.0865,  1.6616, -0.6014,  1.4995],
        [ 0.0865,  1.6616, -0.6014,  1.4995, -0.6014,  1.4995],
        [-0.6014,  1.4995, -0.6014,  1.4995,  0.9249,  0.5281],
        [-0.0261,  0.8148, -0.0261,  0.8148, -0.0261,  0.8148],
        [-0.0261,  0.8148, -0.0261,  0.8148,  0.6220, -0.2713],
        [-0.0261,  0.8148,  0.6220, -0.2713,  0.4949,  0.9084],
        [ 0.6220, -0.2713,  0.4949,  0.9084,  1.2898,  0.4652],
        [ 0.4949,  0.9084,  1.2898,  0.4652, -0.3351,  0.1436],
        [ 1.2898,  0.4652, -0.3351,  0.1436,  1.2898,  0.4652],
        [-0.3351,  0.1436,  1.2898,  0.4652,  0.9249,  0.5281],
        [-0.0261,  0.8148, -0.0261,  0.8148, -0.0261,  0.8148],
        [-0.0261,  0.8148, -0.0261,  0.8148,  0.9249,  0.5281],
        [-0.0261,  0.8148,  0.9249,  0.5281, -0.3351,  0.1436],
        [ 0.9249,  0.5281, -0.3351,  0.1

In [ ]:
# more efficient way to do it by using a view: there is no memory change of the tensor
# the a.storage() stays the same. Changing only attributes to represent the tensor

In [68]:
a = torch.arange(18)
a

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])

In [69]:
a.shape

torch.Size([18])

In [70]:
a.view(2,9)

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8],
        [ 9, 10, 11, 12, 13, 14, 15, 16, 17]])

In [71]:
a.storage()

/var/folders/6x/g47f8b917pv3w7c2_dccwxw40000gn/T/ipykernel_38025/214256462.py:1: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  a.storage()


 0
 1
 2
 3
 4
 5
 6
 7
 8
 9
 10
 11
 12
 13
 14
 15
 16
 17
[torch.storage.TypedStorage(dtype=torch.int64, device=cpu) of size 18]

In [72]:
a.untyped_storage()

 0
 0
 0
 0
 0
 0
 0
 0
 1
 0
 0
 0
 0
 0
 0
 0
 2
 0
 0
 0
 0
 0
 0
 0
 3
 0
 0
 0
 0
 0
 0
 0
 4
 0
 0
 0
 0
 0
 0
 0
 5
 0
 0
 0
 0
 0
 0
 0
 6
 0
 0
 0
 0
 0
 0
 0
 7
 0
 0
 0
 0
 0
 0
 0
 8
 0
 0
 0
 0
 0
 0
 0
 9
 0
 0
 0
 0
 0
 0
 0
 10
 0
 0
 0
 0
 0
 0
 0
 11
 0
 0
 0
 0
 0
 0
 0
 12
 0
 0
 0
 0
 0
 0
 0
 13
 0
 0
 0
 0
 0
 0
 0
 14
 0
 0
 0
 0
 0
 0
 0
 15
 0
 0
 0
 0
 0
 0
 0
 16
 0
 0
 0
 0
 0
 0
 0
 17
 0
 0
 0
 0
 0
 0
 0
[torch.storage.UntypedStorage(device=cpu) of size 144]

In [73]:
emb.shape

torch.Size([32, 3, 2])

In [74]:
emb.view(32,6)

tensor([[-0.0261,  0.8148, -0.0261,  0.8148, -0.0261,  0.8148],
        [-0.0261,  0.8148, -0.0261,  0.8148,  0.0865,  1.6616],
        [-0.0261,  0.8148,  0.0865,  1.6616, -0.6014,  1.4995],
        [ 0.0865,  1.6616, -0.6014,  1.4995, -0.6014,  1.4995],
        [-0.6014,  1.4995, -0.6014,  1.4995,  0.9249,  0.5281],
        [-0.0261,  0.8148, -0.0261,  0.8148, -0.0261,  0.8148],
        [-0.0261,  0.8148, -0.0261,  0.8148,  0.6220, -0.2713],
        [-0.0261,  0.8148,  0.6220, -0.2713,  0.4949,  0.9084],
        [ 0.6220, -0.2713,  0.4949,  0.9084,  1.2898,  0.4652],
        [ 0.4949,  0.9084,  1.2898,  0.4652, -0.3351,  0.1436],
        [ 1.2898,  0.4652, -0.3351,  0.1436,  1.2898,  0.4652],
        [-0.3351,  0.1436,  1.2898,  0.4652,  0.9249,  0.5281],
        [-0.0261,  0.8148, -0.0261,  0.8148, -0.0261,  0.8148],
        [-0.0261,  0.8148, -0.0261,  0.8148,  0.9249,  0.5281],
        [-0.0261,  0.8148,  0.9249,  0.5281, -0.3351,  0.1436],
        [ 0.9249,  0.5281, -0.3351,  0.1

In [77]:
h = torch.tanh(emb.view(-1,6) @ W1 + b1)
h

# emb.view(32,6) can be replaced by emb.view(-1,6) -> torch understand the size automatically

tensor([[ 0.9162,  0.4222, -0.2907,  ...,  0.7293, -0.9986, -0.7639],
        [ 0.9762,  0.9533, -0.3528,  ...,  0.8374, -0.9973, -0.9194],
        [ 0.7627, -0.1802, -0.8560,  ...,  0.9744, -0.9978,  0.0393],
        ...,
        [ 0.7745,  0.9631,  0.9383,  ..., -0.9904, -0.9990, -0.9970],
        [ 0.9576,  0.9946, -0.9197,  ..., -0.8570, -0.8222, -0.9993],
        [ 0.8674,  0.8219, -0.7982,  ...,  0.9545, -0.9935, -0.9952]])

In [78]:
h.shape

torch.Size([32, 100])

In [ ]:
# broadcasting works here too
# emb @ W1 is (32, 100) and b1 is (100) -> align to the right, fake dimension 1, same b1 added to all the rows
# 32, 100
#  1 , 100

In [80]:
# create 2nd and final hidden layer - input: 100 and output 27: possible characters
W2 = torch.randn((100, 27)) 
b2 = torch.randn(27)

In [81]:
logits = h @ W2 + b2

In [82]:
logits.shape

torch.Size([32, 27])

In [83]:
logits

tensor([[ 5.5411e+00, -6.3364e+00,  9.2599e+00,  8.0510e+00, -6.0501e+00,
         -1.2219e+01, -3.7247e+00, -5.8396e+00,  1.7643e+00,  3.7020e+00,
          1.1815e+01, -1.2855e+01, -2.2952e+00,  9.7551e+00, -2.9540e+00,
         -5.2186e+00,  8.9610e+00,  1.1140e+01,  7.6876e+00,  4.2661e+00,
         -6.2710e+00,  6.9720e+00,  1.0786e+01, -1.7108e+01, -7.9457e+00,
         -5.2852e+00,  3.1651e+00],
        [ 5.8587e+00, -1.1542e+01,  5.7123e+00,  5.6661e+00, -6.6847e+00,
         -7.7424e+00, -4.4072e+00, -4.4671e+00,  3.8662e+00,  3.6211e-01,
          1.3512e+01, -1.7510e+01, -1.5485e+00,  1.5210e+01, -4.5631e+00,
         -5.2026e+00,  8.2590e+00,  9.1463e+00,  1.4231e+01,  2.5598e+00,
         -1.6822e+00,  5.4955e+00,  1.1131e+01, -1.8150e+01, -1.2277e+01,
         -1.5928e+00,  3.5747e+00],
        [ 7.6059e+00, -8.0530e+00,  8.2707e+00,  2.1824e+00, -2.7983e+00,
         -3.8383e+00, -5.5468e+00, -4.2334e+00,  1.6357e+01, -6.8564e+00,
          1.3503e+01, -1.2489e+01, -4.24

In [85]:
counts = logits.exp()

In [86]:
prob = counts / counts.sum(1, keepdims=True)

In [87]:
prob.shape

torch.Size([32, 27])

In [89]:
prob[0].sum()

tensor(1.0000)

In [94]:
loss = -prob[torch.arange(32), Y].log().mean()
loss
# for many chars, it does not work well as the network thinks they're very unlikely
# this because the network hasn't been trained yet 
# we want to maximize the prob -> minimize the negative log_loss

tensor(15.1629)

In [91]:
# index in the rows of prob and each row to plug out the prob of the correct char given by Y
torch.arange(32)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31])

In [90]:
# Y is the array created showing the next char we'd like to predict
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])